# Applications of AI in chromatography

**HPLC 2026 — Introduction to Artificial Intelligence for Liquid-phase Separations**

**Prepared for the course by Dr. Tijmen S. Bos.**  
This notebook is part of the course **Introduction to Artificial Intelligence for Liquid-phase Separations** and is intended as browser-based teaching material for the HPLC 2026 course.

**Company affiliation.**  
This material is part of **Innovative Data Evaluation And Separations (IDEAS)**, Dr. Tijmen S. Bos's company for data evaluation and separations education/consultancy. Website: **bos-ideas.com**.

**Ownership and use.**  
The educational content in this notebook belongs to **Dr. Tijmen S. Bos / Innovative Data Evaluation And Separations (IDEAS)**, unless a different source is explicitly indicated. The examples are provided for course participation, demonstration, and educational discussion.

**Educational-use and non-liability statement.**  
The examples in this notebook are simplified teaching simulations. They are not validated analytical methods, software-validation procedures, regulatory guidance, medical/clinical advice, or a substitute for expert chromatographic method development, instrument validation, system suitability testing, or data-integrity review. Any use outside the course context is at the user's own responsibility.

This notebook is a browser-friendly JupyterLite exercise based on the **Applications of AI in Chromatography** part of the course. It covers three application families from the slides:

- **Data processing** — baseline correction, denoising, peak detection, and integration review.
- **Prediction** — retention-time prediction and identification support.
- **Optimization** — method-development decisions and workflow selection, with links to the separate optimization notebook.

The datasets are synthetic, so the exercise is safe to distribute and does not need external files.


## Learning goals

After this notebook, participants should be able to:

1. Map AI applications onto the chromatographic workflow: **data processing**, **prediction**, and **optimization**.
2. Explain why blank runs can be useful for baseline/noise modelling.
3. Distinguish peak detection from peak-integration review.
4. Explain how retention-time prediction and calibration support identification.
5. Combine chromatographic evidence and mass-spectral evidence in a transparent scoring model.
6. Use simple browser-side tools to test their own **SMILES chemical structures** and **chromatogram CSV traces**.
7. Decide when a simple model is preferable to a complex AI model.


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

import json
import math
import uuid
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import HTML, display

RANDOM_STATE = 7
rng = np.random.default_rng(RANDOM_STATE)

# IDEAS/HPLC 2026 palette
IDEAS_CYAN = "#43CEFF"
IDEAS_MAGENTA = "#DB3FC8"
IDEAS_GOLD = "#FFC000"
IDEAS_GREEN = "#92D050"
IDEAS_TEXT = "#16495A"
IDEAS_GRID = "#E5F8FD"
IDEAS_PALE_CYAN = "#D9F7FF"
IDEAS_PALE_MAGENTA = "#F8D8F4"
IDEAS_PALE_GOLD = "#FFF2BF"
IDEAS_PALE_GREEN = "#E7F5D9"

APP_COLORS = [IDEAS_CYAN, IDEAS_MAGENTA, IDEAS_GOLD, IDEAS_GREEN]
APP_CMAP = LinearSegmentedColormap.from_list(
    "ideas_applications",
    [IDEAS_PALE_CYAN, IDEAS_CYAN, IDEAS_MAGENTA, IDEAS_GOLD],
)

plt.rcParams.update({
    "axes.edgecolor": IDEAS_TEXT,
    "axes.labelcolor": IDEAS_TEXT,
    "axes.titlecolor": IDEAS_TEXT,
    "xtick.color": IDEAS_TEXT,
    "ytick.color": IDEAS_TEXT,
    "grid.color": IDEAS_GRID,
    "grid.alpha": 0.75,
    "legend.frameon": True,
})


def gaussian(x, mu, sigma, amp):
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


def chromatographic_baseline(t, rng, strength=1.0):
    """Synthetic baseline drift: low-frequency components plus mild curvature."""
    phase1 = rng.uniform(0, 2*np.pi)
    phase2 = rng.uniform(0, 2*np.pi)
    slope = rng.normal(2.0, 0.8)
    curve = rng.normal(0.18, 0.05) * (t - t.mean()) ** 2
    baseline = (
        34
        + strength * (10*np.sin(0.55*t + phase1) + 6*np.sin(1.1*t + phase2))
        + slope * t
        + curve
    )
    return baseline


def make_chromatogram(seed=1, n=420, peaks=True, peak_load="medium", noise=2.4, baseline_strength=1.0):
    local_rng = np.random.default_rng(seed)
    t = np.linspace(0, 10, n)
    baseline = chromatographic_baseline(t, local_rng, strength=baseline_strength)
    signal = baseline + local_rng.normal(0, noise, size=n)
    true_peaks = []
    if peaks:
        if peak_load == "low":
            specs = [(2.25, 0.10, 85), (5.75, 0.18, 120), (8.20, 0.13, 70)]
        elif peak_load == "high":
            specs = [(1.55, 0.10, 60), (2.35, 0.13, 95), (3.10, 0.11, 62),
                     (5.25, 0.18, 110), (5.75, 0.15, 85), (7.15, 0.13, 70),
                     (8.45, 0.18, 105)]
        elif peak_load == "overlap":
            specs = [(2.25, 0.12, 85), (2.45, 0.13, 70), (5.35, 0.18, 120),
                     (5.62, 0.16, 105), (8.10, 0.18, 75)]
        else:
            specs = [(2.20, 0.12, 90), (4.35, 0.16, 80), (5.75, 0.18, 135), (8.35, 0.15, 70)]
        for mu, sigma, amp in specs:
            jitter = local_rng.normal(0, 0.025)
            amp_j = amp * local_rng.uniform(0.92, 1.08)
            signal += gaussian(t, mu + jitter, sigma, amp_j)
            true_peaks.append({"rt": float(mu + jitter), "width": float(sigma), "height": float(amp_j)})
    return {"time": t, "baseline": baseline, "signal": signal, "peaks": true_peaks}


def make_blank_matrix(n_blanks=30, n=420, noise=2.3, seed=100):
    local_rng = np.random.default_rng(seed)
    t = np.linspace(0, 10, n)
    blanks = []
    for i in range(n_blanks):
        baseline = chromatographic_baseline(t, local_rng, strength=local_rng.uniform(0.75, 1.25))
        blanks.append(baseline + local_rng.normal(0, noise, size=n))
    return t, np.asarray(blanks)


def fit_blank_pca(blanks, n_components=4):
    mean = blanks.mean(axis=0)
    centered = blanks - mean
    # Browser-safe PCA: diagonalize the smaller blank-by-blank covariance matrix
    # instead of running a full SVD on all chromatogram points.
    cov = centered @ centered.T
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = np.argsort(eigvals)[::-1]
    eigvecs = eigvecs[:, order[:n_components]]
    components = eigvecs.T @ centered
    norms = np.linalg.norm(components, axis=1, keepdims=True)
    components = components / np.maximum(norms, 1e-12)
    return {"mean": mean, "components": components}


def predict_blank_pca(model, y):
    centered = y - model["mean"]
    coeff = centered @ model["components"].T
    return model["mean"] + coeff @ model["components"]


_BLANK_MODEL_CACHE = {}

def baseline_application(blank_count=30, n_components=4, peak_load="medium", noise=2.4):
    # Cache the blank model by blank-count and noise only. The SVD is the expensive part;
    # different component counts reuse the same fitted component basis.
    cache_key = (int(blank_count), float(noise))
    if cache_key not in _BLANK_MODEL_CACHE:
        t, blanks = make_blank_matrix(n_blanks=blank_count, noise=noise, seed=200 + blank_count)
        _BLANK_MODEL_CACHE[cache_key] = fit_blank_pca(blanks, n_components=8)
    full_model = _BLANK_MODEL_CACHE[cache_key]
    model = {
        "mean": full_model["mean"],
        "components": full_model["components"][:int(n_components)],
    }
    sample = make_chromatogram(seed=42, peaks=True, peak_load=peak_load, noise=noise)
    pred = predict_blank_pca(model, sample["signal"])
    corrected = sample["signal"] - pred
    rmse = float(np.sqrt(np.mean((pred - sample["baseline"]) ** 2)))
    peak_area_est = float(np.trapezoid(np.clip(corrected, 0, None), sample["time"]))
    return {"time": sample["time"], "signal": sample["signal"], "baseline": sample["baseline"], "pred": pred, "corrected": corrected, "rmse": rmse, "area": peak_area_est}


def moving_average(y, window=9):
    window = max(3, int(window) | 1)
    pad = window // 2
    padded = np.pad(y, pad, mode="edge")
    kernel = np.ones(window) / window
    return np.convolve(padded, kernel, mode="valid")


def peak_detection_case(overlap="medium", sensitivity=0.55, noise=2.5):
    peak_load = {"low":"low", "medium":"medium", "high":"overlap"}.get(overlap, "medium")
    case = make_chromatogram(seed=66, peaks=True, peak_load=peak_load, noise=noise)
    t, y, baseline = case["time"], case["signal"], case["baseline"]
    smooth = moving_average(y, 11)
    residual = smooth - baseline
    threshold = 18 + (1 - sensitivity) * 50
    candidates = []
    for i in range(2, len(t)-2):
        if residual[i] > threshold and residual[i] >= residual[i-1] and residual[i] >= residual[i+1]:
            # suppress near-duplicates
            if candidates and (t[i] - candidates[-1]["rt"]) < 0.18:
                if residual[i] > candidates[-1]["height"]:
                    candidates[-1] = {"rt": float(t[i]), "height": float(residual[i])}
            else:
                candidates.append({"rt": float(t[i]), "height": float(residual[i])})
    true_rts = np.array([p["rt"] for p in case["peaks"]])
    tp = 0
    fp = 0
    labels = []
    for cand in candidates:
        is_true = len(true_rts) and np.min(np.abs(true_rts - cand["rt"])) < 0.16
        labels.append(bool(is_true))
        tp += int(is_true)
        fp += int(not is_true)
    matched_true = set()
    for cand in candidates:
        if len(true_rts):
            j = int(np.argmin(np.abs(true_rts - cand["rt"])))
            if abs(true_rts[j] - cand["rt"]) < 0.16:
                matched_true.add(j)
    fn = len(true_rts) - len(matched_true)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    return {"time": t, "signal": y, "baseline": baseline, "candidates": candidates, "labels": labels, "true_peaks": case["peaks"], "precision": float(precision), "recall": float(recall), "threshold": float(threshold)}


def make_retention_data(n=240, seed=88):
    local_rng = np.random.default_rng(seed)
    logp = local_rng.uniform(-1.2, 5.2, n)
    pka = local_rng.uniform(2, 11, n)
    psa = local_rng.uniform(15, 135, n)
    mass = local_rng.uniform(110, 620, n)
    charge = local_rng.integers(0, 3, n)
    X = np.column_stack([logp, pka, psa, mass, charge])
    rt = (
        2.0 + 0.85*logp + 0.006*mass - 0.025*psa + 0.28*np.maximum(pka - 7, 0)
        - 0.65*charge + 0.25*np.sin(1.4*logp) + local_rng.normal(0, 0.28, n)
    )
    rt = np.clip(rt, 0.6, 11.5)
    return X, rt


def design_matrix(X):
    X = np.asarray(X)
    # Scale approximately to manageable ranges.
    Z = X.copy().astype(float)
    Z[:,0] = (Z[:,0] - 2.0) / 2.2
    Z[:,1] = (Z[:,1] - 6.5) / 3.0
    Z[:,2] = (Z[:,2] - 70) / 45
    Z[:,3] = (Z[:,3] - 350) / 170
    Z[:,4] = (Z[:,4] - 1.0)
    feats = [np.ones(len(Z))]
    feats.extend([Z[:,i] for i in range(Z.shape[1])])
    feats.extend([Z[:,i]**2 for i in range(Z.shape[1])])
    feats.append(Z[:,0] * Z[:,2])
    feats.append(Z[:,0] * Z[:,3])
    feats.append(np.sin(1.3*Z[:,0]))
    return np.column_stack(feats)


def fit_ridge(Phi, y, lam=1e-2):
    A = Phi.T @ Phi + lam*np.eye(Phi.shape[1])
    A[0,0] -= lam
    return np.linalg.solve(A, Phi.T @ y)


def retention_prediction_case(train_size=60, calibrants=5):
    X, rt = make_retention_data(n=260)
    idx = np.arange(len(rt))
    rng_local = np.random.default_rng(44 + train_size + calibrants)
    rng_local.shuffle(idx)
    train_idx = idx[:train_size]
    test_idx = idx[160:240]
    Phi_train = design_matrix(X[train_idx])
    beta = fit_ridge(Phi_train, rt[train_idx])
    pred = design_matrix(X[test_idx]) @ beta
    true = rt[test_idx]
    if calibrants > 0:
        cal_idx = test_idx[:calibrants]
        raw_cal = design_matrix(X[cal_idx]) @ beta
        true_cal = rt[cal_idx]
        if calibrants >= 2:
            slope, intercept = np.polyfit(raw_cal, true_cal, 1)
        else:
            slope, intercept = 1.0, float(true_cal[0] - raw_cal[0])
        pred = slope * pred + intercept
    mae = float(np.mean(np.abs(pred - true)))
    return {"true": true, "pred": pred, "mae": mae, "train_size": train_size, "calibrants": calibrants}


def identification_case(rt_weight=0.5, ms_noise="medium", n_candidates=8):
    rng_local = np.random.default_rng(909 + int(rt_weight*10) + n_candidates)
    true_rt = 5.85
    names = ["candidate_" + chr(65+i) for i in range(n_candidates)]
    true_index = min(2, n_candidates-1)
    predicted_rt = rng_local.normal(true_rt + 0.8, 1.4, n_candidates)
    ms_similarity = rng_local.uniform(0.45, 0.86, n_candidates)
    # make one correct candidate with high spectral similarity and plausible RT
    predicted_rt[true_index] = true_rt + rng_local.normal(0, 0.22 if ms_noise != "high" else 0.42)
    ms_similarity[true_index] = 0.92 - (0.05 if ms_noise == "high" else 0.0)
    if n_candidates > 5:
        # add a spectral decoy that is good in MS but bad in RT
        decoy = min(5, n_candidates-1)
        ms_similarity[decoy] = 0.94
        predicted_rt[decoy] = true_rt + 1.9
    rt_sigma = 0.55
    rt_score = np.exp(-0.5 * ((predicted_rt - true_rt) / rt_sigma) ** 2)
    score = (1 - rt_weight) * ms_similarity + rt_weight * rt_score
    order = np.argsort(score)[::-1]
    return {
        "names": [names[i] for i in order],
        "rt_pred": [float(predicted_rt[i]) for i in order],
        "ms": [float(ms_similarity[i]) for i in order],
        "rt_score": [float(rt_score[i]) for i in order],
        "score": [float(score[i]) for i in order],
        "true_name": names[true_index],
        "true_rt": true_rt,
    }

print("Application notebook helper functions loaded.")


## 2. Application map from the slides

The slides group chromatographic AI applications into three broad families: **data processing**, **prediction**, and **optimization**. In this notebook, the code examples focus mainly on data processing and prediction, because the optimization family is covered separately in the optimization notebook.


In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.5))
ax.axis("off")
blocks = [
    (0.03, 0.20, 0.28, 0.62, "DATA PROCESSING", ["Baseline correction", "Noise removal", "Peak detection", "Peak integration review"]),
    (0.36, 0.20, 0.28, 0.62, "PREDICTION", ["Retention time", "Candidate identification", "Mass-spectral evidence", "Confidence scoring"]),
    (0.69, 0.20, 0.28, 0.62, "OPTIMIZATION", ["Method development", "Gradient selection", "Experimental budget", "Closed-loop decisions"]),
]
for k, (x, y0, w, h, title, items) in enumerate(blocks):
    color = [IDEAS_PALE_CYAN, IDEAS_PALE_MAGENTA, IDEAS_PALE_GOLD][k]
    edge = [IDEAS_CYAN, IDEAS_MAGENTA, IDEAS_GOLD][k]
    rect = plt.Rectangle((x, y0), w, h, facecolor=color, edgecolor=edge, linewidth=2.2)
    ax.add_patch(rect)
    ax.text(x+w/2, y0+h-0.10, title, ha="center", va="center", fontsize=13, fontweight="bold", color=IDEAS_TEXT)
    for j, item in enumerate(items):
        ax.text(x+0.04, y0+h-0.22-0.10*j, "• " + item, ha="left", va="center", fontsize=10.5, color=IDEAS_TEXT)
ax.text(0.5, 0.93, "Applications of AI across the chromatographic workflow", ha="center", va="center", fontsize=15, fontweight="bold", color=IDEAS_TEXT)
plt.show()


## 3. Literature-informed example selection

The slide sequence from slide 90 onward covers AI applications in chromatography: baseline correction, noise removal, peak detection, peak integration, identification, retention-time prediction, mass spectra, and method development. The notebook follows that structure with browser-safe teaching analogues of the published examples.

A recent-literature scan suggests that the slide examples are well chosen, but three practical teaching additions are especially useful: **manual data import**, **retention-time calibration**, and **transparent evidence scoring**. These additions make the examples easier to connect to real student data without requiring specialized packages such as RDKit, TensorFlow, PyTorch, or vendor-specific chromatography software.

| Application family | Literature signal | Teaching implementation in this notebook |
|---|---|---|
| Baseline correction / denoising | Deep convolutional autoencoders have been used to remove baseline drift and noise from chromatograms. Kensert et al., *J. Chromatogr. A* 2021, 1646, 462093. | A light blank-trained PCA baseline model. It is not a deep autoencoder, but it teaches the same principle: learn background structure from representative blank/background traces. |
| Peak detection | CNN-based peak detection has been demonstrated for reversed-phase LC. The slides cite Kensert et al., *J. Chromatogr. A* 2022, 1672, 463005. | A browser-safe candidate detector with adjustable sensitivity, precision, and recall. |
| Peak integration review | Recent work uses supervised learning to reduce manual targeted LC-MS/MS peak-integration review workload. | A review-oriented peak-candidate exercise that separates detection sensitivity from validation responsibility. |
| Retention-time prediction / QSRR | Recent work includes graph neural networks, transformer/language-model approaches, transfer learning, and calibration for LC-MS retention time. | A structure-to-retention teaching model using simple SMILES-derived descriptors and explicit calibration. |
| Identification support | Recent AI/MS literature emphasizes combining spectral evidence, retention-time prediction, and uncertainty. | A transparent evidence-fusion example that combines retention-time agreement with spectral similarity. |
| Method development | Recent LC-method-development work includes active learning, Bayesian optimization, reinforcement learning, retention modelling, and multi-fidelity optimization. | The optimization family is summarized here and explored in more detail in the separate optimization notebook. |

The student-input section intentionally uses simple, transparent algorithms. This is by design: students can see what assumptions the model makes, and the examples remain practical in JupyterLite.


## 3b. Coverage of the slide examples

The notebook maps the slide examples to browser-safe exercises as follows:

| Slide topic | Notebook example |
|---|---|
| Data processing: baseline correction, noise removal, peak detection, peak integration | Blank-trained baseline model; peak detection and review; user chromatogram analyzer |
| Autoencoder baseline correction | PCA bottleneck model trained on blanks as a lightweight autoencoder analogue |
| CNN/DNN peak detection | Thresholded local-candidate detector with precision/recall interpretation as a lightweight CNN analogue |
| QSRR and neural-network/graph representations | SMILES-to-descriptor retention prediction exercise |
| Mass spectra / identification support | Transparent evidence fusion with retention-time and spectral evidence |
| Reinforcement learning, Bayesian optimization, multi-fidelity optimization | Summarized here; developed further in the separate intelligent-machine-learning and optimization notebooks |


## 4. Application example: blank-trained baseline correction

In a chromatographic laboratory, blank runs can contain useful information about detector noise, slow drift, pump-related effects, and other non-analyte structure. A model trained only on blanks should learn the background but should not learn analyte peaks as part of the baseline.

This example uses a light PCA model trained on simulated blank chromatograms. It is a teaching analogue of the deeper autoencoder idea discussed in the slides and recent literature.


In [ ]:
result = baseline_application(blank_count=30, n_components=4, peak_load="medium", noise=2.4)
t, raw, true_base, pred_base, corrected = result["time"], result["signal"], result["baseline"], result["pred"], result["corrected"]

fig, axes = plt.subplots(2, 1, figsize=(10, 6.4), sharex=True)
axes[0].plot(t, raw, color=IDEAS_TEXT, linewidth=1.4, label="sample chromatogram")
axes[0].plot(t, true_base, color=IDEAS_GREEN, linewidth=2.0, label="true baseline (known only in simulation)")
axes[0].plot(t, pred_base, color=IDEAS_MAGENTA, linewidth=2.0, label="blank-trained baseline model")
axes[0].set_ylabel("Signal (mAU)")
axes[0].set_title("Baseline model trained from blank chromatograms")
axes[0].grid(True, linewidth=0.6)
axes[0].legend(fontsize=8)

axes[1].plot(t, corrected, color=IDEAS_CYAN, linewidth=1.5, label="baseline-corrected signal")
axes[1].axhline(0, color=IDEAS_TEXT, linewidth=0.8, alpha=0.5)
axes[1].set_xlabel("Time (min)")
axes[1].set_ylabel("Corrected signal (mAU)")
axes[1].set_title(f"Corrected chromatogram; baseline RMSE = {result['rmse']:.2f} mAU")
axes[1].grid(True, linewidth=0.6)
axes[1].legend(fontsize=8)
plt.subplots_adjust(hspace=0.26)
plt.show()


**Interpretation.**  
A blank-trained background model is useful when the non-analyte signal is reproducible enough to learn. However, a model with too much flexibility can begin to treat broad or overlapping analyte peaks as part of the baseline. This is why blanks, validation samples, and analyst review remain important.


## 5. Application example: peak detection and integration review

Peak detection and integration are not the same task. Peak detection asks: **where are plausible chromatographic peaks?** Integration review asks: **which candidate peaks are acceptable for quantification?** AI can help reduce manual review workload, but the output still requires validation and traceability.


In [ ]:
case = peak_detection_case(overlap="high", sensitivity=0.62, noise=2.6)
t, y, baseline = case["time"], case["signal"], case["baseline"]

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(t, y, color=IDEAS_TEXT, linewidth=1.2, label="chromatogram")
ax.plot(t, baseline, color=IDEAS_GREEN, linewidth=1.7, label="baseline")
for p in case["true_peaks"]:
    ax.axvspan(p["rt"] - 0.07, p["rt"] + 0.07, color=IDEAS_PALE_GREEN, alpha=0.8)
for cand, is_true in zip(case["candidates"], case["labels"]):
    ax.scatter(cand["rt"], baseline[np.argmin(np.abs(t-cand["rt"]))] + cand["height"],
               s=80, marker="o", color=IDEAS_CYAN if is_true else IDEAS_MAGENTA,
               edgecolor="white", linewidth=1.0)
ax.set_xlabel("Time (min)")
ax.set_ylabel("Signal (mAU)")
ax.set_title(f"Peak candidates; precision = {case['precision']:.2f}, recall = {case['recall']:.2f}")
ax.grid(True, linewidth=0.6)
ax.legend(fontsize=8)
plt.show()

summary = pd.DataFrame({
    "metric": ["candidate count", "precision", "recall", "detection threshold"],
    "value": [len(case["candidates"]), f"{case['precision']:.2f}", f"{case['recall']:.2f}", f"{case['threshold']:.1f} mAU above baseline"],
})
display(summary)


**Interpretation.**  
Low thresholds increase recall but may create many false candidates. High thresholds reduce review burden but can miss low-abundance compounds. For quantitative use, a model should usually be positioned as an **assistant for review prioritization**, not as an unchecked replacement for validated integration rules.


## 6. Application example: retention-time prediction and calibration

Retention-time prediction can support method development, compound annotation, and LC-MS data review. The practical problem is that retention times are instrument-, column-, and method-dependent. External calibrants or transfer models can therefore be essential.


In [ ]:
rt_case = retention_prediction_case(train_size=60, calibrants=8)
true, pred = rt_case["true"], rt_case["pred"]

fig, ax = plt.subplots(figsize=(6.2, 5.4))
ax.scatter(true, pred, s=36, color=IDEAS_CYAN, edgecolor="white", linewidth=0.6, alpha=0.86)
lo = min(true.min(), pred.min()) - 0.4
hi = max(true.max(), pred.max()) + 0.4
ax.plot([lo, hi], [lo, hi], color=IDEAS_MAGENTA, linewidth=2, label="ideal")
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_xlabel("Observed retention time (min)")
ax.set_ylabel("Predicted retention time (min)")
ax.set_title(f"Retention-time prediction after calibration; MAE = {rt_case['mae']:.2f} min")
ax.grid(True, linewidth=0.6)
ax.legend(fontsize=8)
plt.show()


**Interpretation.**  
Model complexity is only one part of retention prediction. Calibration points can correct systematic shifts caused by the instrument, column, gradient delay, dwell volume, or mobile-phase differences. This is why retention-time prediction is often most useful when combined with a transfer/calibration strategy.


## 7. Application example: identification support with spectral and RT evidence

For LC-MS, identification confidence can improve when orthogonal evidence sources are combined. This toy example ranks candidate identities by combining a mass-spectral similarity score with agreement between observed and predicted retention time.


In [ ]:
id_case = identification_case(rt_weight=0.45, ms_noise="medium", n_candidates=10)
labels = id_case["names"]
scores = id_case["score"]
colors = [IDEAS_GREEN if name == id_case["true_name"] else IDEAS_CYAN for name in labels]

fig, ax = plt.subplots(figsize=(9.5, 4.8))
ypos = np.arange(len(labels))
ax.barh(ypos, scores, color=colors, edgecolor="white", linewidth=0.7)
ax.set_yticks(ypos)
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel("Combined identification score")
ax.set_title("Candidate ranking from MS similarity and retention-time agreement")
ax.grid(True, axis="x", linewidth=0.6)
plt.show()

candidate_table = pd.DataFrame({
    "candidate": labels,
    "predicted RT (min)": [f"{x:.2f}" for x in id_case["rt_pred"]],
    "MS similarity": [f"{x:.2f}" for x in id_case["ms"]],
    "RT score": [f"{x:.2f}" for x in id_case["rt_score"]],
    "combined score": [f"{x:.2f}" for x in id_case["score"]],
})
display(candidate_table)


**Interpretation.**  
A high spectral score alone can be misleading if retention behaviour is incompatible with the candidate. Conversely, a plausible retention time does not prove the structure. A transparent combined score is useful for teaching because it shows how AI can support identification without hiding the assumptions.


## 8. Interactive applications exercise with built-in scenarios

Use the interactive exercise below to compare application modules using built-in teaching scenarios. The controls are browser-side and do not require `ipywidgets`.

The panel now shows three explicit questions for the selected application module. Suggested answers map one-to-one to Q1, Q2, and Q3; students can use them as a starting point but should still explain the result in chromatographic language.


In [ ]:
# Interactive applications exercise: browser-side controls, no ipywidgets.

from IPython.display import HTML, display

# Precompute compact payloads so the dashboard remains responsive in JupyterLite.
payload = {
    "baseline": {},
    "peaks": {},
    "retention": {},
    "identification": {},
}

for blanks in [5, 20, 80]:
    for comps in [2, 4, 8]:
        for load in ["low", "medium", "high"]:
            r = baseline_application(blank_count=blanks, n_components=comps, peak_load=load, noise=2.4)
            key = f"blanks={blanks}|components={comps}|load={load}"
            payload["baseline"][key] = {
                "time": [float(x) for x in r["time"]],
                "signal": [float(x) for x in r["signal"]],
                "baseline": [float(x) for x in r["baseline"]],
                "pred": [float(x) for x in r["pred"]],
                "corrected": [float(x) for x in r["corrected"]],
                "rmse": float(r["rmse"]),
                "area": float(r["area"]),
            }

for overlap in ["low", "medium", "high"]:
    for sensitivity in [0.35, 0.55, 0.75]:
        r = peak_detection_case(overlap=overlap, sensitivity=sensitivity, noise=2.6)
        key = f"overlap={overlap}|sensitivity={sensitivity:.2f}"
        payload["peaks"][key] = {
            "time": [float(x) for x in r["time"]],
            "signal": [float(x) for x in r["signal"]],
            "baseline": [float(x) for x in r["baseline"]],
            "candidates": r["candidates"],
            "labels": r["labels"],
            "true_peaks": r["true_peaks"],
            "precision": float(r["precision"]),
            "recall": float(r["recall"]),
            "threshold": float(r["threshold"]),
        }

for train_size in [20, 60, 150]:
    for cal in [0, 5, 20]:
        r = retention_prediction_case(train_size=train_size, calibrants=cal)
        key = f"train={train_size}|cal={cal}"
        payload["retention"][key] = {
            "true": [float(x) for x in r["true"]],
            "pred": [float(x) for x in r["pred"]],
            "mae": float(r["mae"]),
        }

for rt_weight in [0.0, 0.45, 0.75]:
    for ms_noise in ["low", "medium", "high"]:
        for n_candidates in [4, 10, 20]:
            r = identification_case(rt_weight=rt_weight, ms_noise=ms_noise, n_candidates=n_candidates)
            key = f"rtw={rt_weight:.2f}|noise={ms_noise}|n={n_candidates}"
            payload["identification"][key] = r

container_id = "applications_dashboard_" + uuid.uuid4().hex[:8]
payload_json = json.dumps(payload)

html = f"""
<div id="{container_id}" class="ideas-dashboard">
  <style>
    #{container_id} {{
      --cyan: {IDEAS_CYAN};
      --magenta: {IDEAS_MAGENTA};
      --gold: {IDEAS_GOLD};
      --green: {IDEAS_GREEN};
      --text: {IDEAS_TEXT};
      --grid: {IDEAS_GRID};
      --pale-gold: {IDEAS_PALE_GOLD};
      font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
      color: var(--text);
      width: 100%;
      max-width: 1180px;
      border: 2px solid var(--cyan);
      border-radius: 18px;
      padding: 16px;
      box-sizing: border-box;
      background: #ffffff;
      overflow: visible !important;
    }}
    #{container_id} .title {{ font-size: 20px; font-weight: 800; margin-bottom: 4px; }}
    #{container_id} .subtitle {{ font-size: 13px; margin-bottom: 14px; opacity: 0.85; }}
    #{container_id} .layout {{ display: grid; grid-template-columns: 270px minmax(480px, 1fr) 300px; gap: 14px; align-items: start; }}
    #{container_id} .card {{ border: 1px solid #d8eef5; border-radius: 14px; padding: 12px; background: #fbfdfe; box-sizing: border-box; }}
    #{container_id} label {{ display: block; font-weight: 700; font-size: 12px; margin-top: 10px; margin-bottom: 4px; }}
    #{container_id} select, #{container_id} input[type=range] {{ width: 100%; }}
    #{container_id} canvas {{ width: 100%; height: 430px; border: 1px solid #d8eef5; border-radius: 14px; background: white; }}
    #{container_id} .metric {{ font-size: 28px; font-weight: 800; color: var(--magenta); margin-top: 4px; }}
    #{container_id} .small {{ font-size: 12px; line-height: 1.35; }}
    #{container_id} textarea {{ width: 100%; min-height: 72px; box-sizing: border-box; border-radius: 10px; border: 1px solid #c9dfe7; padding: 8px; font-family: inherit; font-size: 12px; }}
    #{container_id} button {{ border: none; border-radius: 10px; padding: 8px 10px; margin: 4px 4px 4px 0; font-weight: 700; cursor: pointer; color: var(--text); background: var(--cyan); }}
    #{container_id} button.secondary {{ background: var(--gold); }}
    #{container_id} .questions {{ border: 1px solid #d8eef5; background: #ffffff; border-radius: 12px; padding: 10px; margin: 10px 0; font-size: 12px; line-height: 1.35; }}
    #{container_id} .questions ol, #{container_id} .answers ol {{ margin-top: 6px; padding-left: 20px; }}
    #{container_id} .answers {{ display:none; border: 2px solid var(--gold); background: var(--pale-gold); border-radius: 12px; padding: 10px; margin-top: 10px; font-size: 12px; line-height: 1.35; }}
    #{container_id} .hidden {{ display: none; }}
    #{container_id} .legend-row {{ display:flex; gap:8px; flex-wrap:wrap; font-size:11px; margin-top:8px; }}
    #{container_id} .dot {{ width:10px; height:10px; border-radius:50%; display:inline-block; margin-right:4px; }}
    @media (max-width: 1050px) {{
      #{container_id} .layout {{ grid-template-columns: 1fr; }}
      #{container_id} canvas {{ height: 420px; }}
    }}
  </style>

  <div class="title">Interactive applications exercise</div>
  <div class="subtitle">Browser-side dashboard for data processing, prediction, and identification examples. IDEAS — bos-ideas.com.</div>

  <div class="layout">
    <div class="card">
      <label>Application module</label>
      <select data-control="module">
        <option value="baseline">Blank-trained baseline correction</option>
        <option value="peaks">Peak detection and review</option>
        <option value="retention">Retention-time prediction</option>
        <option value="identification">Identification scoring</option>
      </select>

      <div data-panel="baseline">
        <label>Number of blank runs</label>
        <select data-control="blanks"><option>5</option><option selected>20</option><option>80</option></select>
        <label>Baseline model components</label>
        <select data-control="components"><option>2</option><option selected>4</option><option>8</option></select>
        <label>Peak load</label>
        <select data-control="load"><option>low</option><option selected>medium</option><option>high</option></select>
      </div>

      <div data-panel="peaks" class="hidden">
        <label>Peak overlap</label>
        <select data-control="overlap"><option>low</option><option selected>medium</option><option>high</option></select>
        <label>Detector sensitivity</label>
        <select data-control="sensitivity"><option value="0.35">conservative</option><option value="0.55" selected>balanced</option><option value="0.75">sensitive</option></select>
      </div>

      <div data-panel="retention" class="hidden">
        <label>Training-set size</label>
        <select data-control="train"><option>20</option><option selected>60</option><option>150</option></select>
        <label>External calibrants</label>
        <select data-control="cal"><option>0</option><option selected>5</option><option>20</option></select>
      </div>

      <div data-panel="identification" class="hidden">
        <label>Retention-time weight</label>
        <select data-control="rtw"><option value="0.00">0.00: MS only</option><option value="0.45" selected>0.45: balanced</option><option value="0.75">0.75: RT-heavy</option></select>
        <label>MS uncertainty</label>
        <select data-control="msnoise"><option>low</option><option selected>medium</option><option>high</option></select>
        <label>Candidate count</label>
        <select data-control="candidates"><option>4</option><option selected>10</option><option>20</option></select>
      </div>
    </div>

    <div class="card">
      <canvas width="720" height="430" data-role="plot"></canvas>
      <div class="legend-row" data-role="legend"></div>
    </div>

    <div class="card">
      <div class="small" data-role="description"></div>
      <div class="metric" data-role="metric"></div>
      <div class="small" data-role="interpretation"></div>
      <div class="questions" data-role="questions"></div>
      <div class="answers" data-role="suggested"></div>
      <button class="secondary" data-action="showAnswers">Show suggested answers</button>
      <button class="secondary" data-action="useAnswers">Use suggested answers in boxes</button>
      <label>Answer to Q1</label><textarea data-answer="0" placeholder="Write your answer to Q1 here..."></textarea>
      <label>Answer to Q2</label><textarea data-answer="1" placeholder="Write your answer to Q2 here..."></textarea>
      <label>Answer to Q3</label><textarea data-answer="2" placeholder="Write your answer to Q3 here..."></textarea>
      <button data-action="copy">Copy answers</button>
      <button data-action="download">Download answers</button>
    </div>
  </div>
</div>

<script>
(function() {{
  const root = document.getElementById("{container_id}");
  const payload = {payload_json};
  const colors = {{ cyan: "{IDEAS_CYAN}", magenta: "{IDEAS_MAGENTA}", gold: "{IDEAS_GOLD}", green: "{IDEAS_GREEN}", text: "{IDEAS_TEXT}", grid: "{IDEAS_GRID}" }};
  const questionSets = {{
    baseline: [
      "What is the AI model trying to learn in the blank-trained baseline-correction example?",
      "How do the number of blank runs, model components, and peak load affect the reliability of the correction?",
      "What validation or manual review would be needed before using this correction in a quantitative workflow?"
    ],
    peaks: [
      "How do peak overlap and detector sensitivity affect precision and recall?",
      "Which error type is more problematic for this chromatographic task: false positives or false negatives? Explain why.",
      "How should AI-supported peak detection be integrated into a validated review workflow?"
    ],
    retention: [
      "How do training-set size and external calibrants affect the retention-time prediction error?",
      "Why should predicted retention time be treated as orthogonal evidence rather than as a standalone identification result?",
      "What would make a compound or method condition outside the useful domain of this retention-time model?"
    ],
    identification: [
      "Which candidate is ranked highest, and what evidence contributes to that ranking?",
      "How do retention-time weight, MS uncertainty, and candidate count influence the risk of a wrong assignment?",
      "What additional evidence or validation would be required before reporting an identification?"
    ]
  }};

  function expandOutput() {{
    const outputSelectors = [".jp-OutputArea-output", ".jp-RenderedHTMLCommon", ".output_html", ".output_subarea"];
    outputSelectors.forEach(sel => {{
      const wrapper = root.closest(sel);
      if (wrapper && wrapper.style) {{ wrapper.style.maxHeight = "none"; wrapper.style.overflow = "visible"; }}
    }});
    document.documentElement.style.overflow = "";
    document.body.style.overflow = "";
  }}
  expandOutput();

  const get = (selector) => root.querySelector(selector);
  const getAll = (selector) => Array.from(root.querySelectorAll(selector));
  const canvas = get('[data-role="plot"]');
  const ctx = canvas.getContext('2d');

  function control(name) {{ return get(`[data-control="${{name}}"]`).value; }}
  function clear() {{ ctx.clearRect(0,0,canvas.width,canvas.height); ctx.fillStyle = '#fff'; ctx.fillRect(0,0,canvas.width,canvas.height); }}
  function ranges(xs, ys) {{
    const xmin = Math.min(...xs), xmax = Math.max(...xs), ymin = Math.min(...ys), ymax = Math.max(...ys);
    const xp = Math.max((xmax-xmin)*0.06, 0.1), yp = Math.max((ymax-ymin)*0.10, 1);
    return [xmin-xp, xmax+xp, ymin-yp, ymax+yp];
  }}
  function mapper(xs, ys, left=58, top=28, right=24, bottom=48) {{
    const [xmin,xmax,ymin,ymax] = ranges(xs, ys);
    const w = canvas.width-left-right, h = canvas.height-top-bottom;
    return {{
      x: v => left + (v-xmin)/(xmax-xmin)*w,
      y: v => top + h - (v-ymin)/(ymax-ymin)*h,
      left, top, w, h, xmin, xmax, ymin, ymax
    }};
  }}
  function grid(mp, xlabel, ylabel) {{
    ctx.strokeStyle = colors.grid; ctx.lineWidth = 1;
    ctx.font = '11px system-ui'; ctx.fillStyle = colors.text;
    for (let i=0; i<=5; i++) {{
      const x = mp.left + i*mp.w/5; ctx.beginPath(); ctx.moveTo(x, mp.top); ctx.lineTo(x, mp.top+mp.h); ctx.stroke();
      const val = mp.xmin + i*(mp.xmax-mp.xmin)/5; ctx.fillText(val.toFixed(1), x-12, mp.top+mp.h+18);
      const y = mp.top + i*mp.h/5; ctx.beginPath(); ctx.moveTo(mp.left, y); ctx.lineTo(mp.left+mp.w, y); ctx.stroke();
      const yv = mp.ymax - i*(mp.ymax-mp.ymin)/5; ctx.fillText(yv.toFixed(0), 8, y+4);
    }}
    ctx.strokeStyle = colors.text; ctx.lineWidth = 1.2; ctx.strokeRect(mp.left, mp.top, mp.w, mp.h);
    ctx.fillText(xlabel, mp.left + mp.w/2 - 40, canvas.height-12);
    ctx.save(); ctx.translate(14, mp.top+mp.h/2+40); ctx.rotate(-Math.PI/2); ctx.fillText(ylabel, 0, 0); ctx.restore();
  }}
  function line(mp, xs, ys, color, width=2) {{
    ctx.strokeStyle = color; ctx.lineWidth = width; ctx.beginPath();
    xs.forEach((x,i)=>{{ const px=mp.x(x), py=mp.y(ys[i]); if(i===0) ctx.moveTo(px,py); else ctx.lineTo(px,py); }});
    ctx.stroke();
  }}
  function scatter(mp, xs, ys, color, r=4) {{
    ctx.fillStyle = color; ctx.strokeStyle = '#fff';
    xs.forEach((x,i)=>{{ ctx.beginPath(); ctx.arc(mp.x(x), mp.y(ys[i]), r, 0, 2*Math.PI); ctx.fill(); ctx.stroke(); }});
  }}
  function setLegend(items) {{
    get('[data-role="legend"]').innerHTML = items.map(it => `<span><span class="dot" style="background:${{it.color}}"></span>${{it.label}}</span>`).join('');
  }}

  function baselineKey() {{ return `blanks=${{control('blanks')}}|components=${{control('components')}}|load=${{control('load')}}`; }}
  function peaksKey() {{ return `overlap=${{control('overlap')}}|sensitivity=${{Number(control('sensitivity')).toFixed(2)}}`; }}
  function retentionKey() {{ return `train=${{control('train')}}|cal=${{control('cal')}}`; }}
  function identificationKey() {{ return `rtw=${{Number(control('rtw')).toFixed(2)}}|noise=${{control('msnoise')}}|n=${{control('candidates')}}`; }}

  function drawBaseline() {{
    const r = payload.baseline[baselineKey()]; clear();
    const ys = r.signal.concat(r.baseline).concat(r.pred);
    const mp = mapper(r.time, ys); grid(mp, 'Time (min)', 'Signal (mAU)');
    line(mp, r.time, r.signal, colors.text, 1.3);
    line(mp, r.time, r.baseline, colors.green, 2.2);
    line(mp, r.time, r.pred, colors.magenta, 2.2);
    setLegend([{{color:colors.text,label:'sample'}}, {{color:colors.green,label:'true baseline'}}, {{color:colors.magenta,label:'blank-trained model'}}]);
    get('[data-role="description"]').textContent = 'Blank-trained baseline correction';
    get('[data-role="metric"]').textContent = `${{r.rmse.toFixed(2)}} mAU`;
    get('[data-role="interpretation"]').textContent = `Baseline RMSE. Increasing blank count usually stabilizes the background model; too many components can start to follow analyte peaks.`;
    return [
      'The model is trained on blanks, so it learns low-frequency background and noise structure rather than analyte peaks.',
      `The baseline RMSE is ${{r.rmse.toFixed(2)}} mAU for the selected settings. This should be interpreted relative to peak height and quantification limits.`,
      'More flexible background models are not automatically better; excessive flexibility may subtract broad peaks as if they were baseline.'
    ];
  }}

  function drawPeaks() {{
    const r = payload.peaks[peaksKey()]; clear();
    const mp = mapper(r.time, r.signal.concat(r.baseline)); grid(mp, 'Time (min)', 'Signal (mAU)');
    line(mp, r.time, r.signal, colors.text, 1.2); line(mp, r.time, r.baseline, colors.green, 1.8);
    r.true_peaks.forEach(p => {{ ctx.fillStyle = 'rgba(146,208,80,0.20)'; const x=mp.x(p.rt); ctx.fillRect(x-5, mp.top, 10, mp.h); }});
    r.candidates.forEach((c,i) => {{ const idx = r.time.reduce((best,v,j)=> Math.abs(v-c.rt)<Math.abs(r.time[best]-c.rt)?j:best,0); ctx.fillStyle = r.labels[i] ? colors.cyan : colors.magenta; ctx.strokeStyle = '#fff'; ctx.beginPath(); ctx.arc(mp.x(c.rt), mp.y(r.baseline[idx]+c.height), 6, 0, 2*Math.PI); ctx.fill(); ctx.stroke(); }});
    setLegend([{{color:colors.text,label:'signal'}}, {{color:colors.green,label:'baseline'}}, {{color:colors.cyan,label:'true candidate'}}, {{color:colors.magenta,label:'false candidate'}}]);
    get('[data-role="description"]').textContent = 'Peak detection and review';
    get('[data-role="metric"]').textContent = `P=${{r.precision.toFixed(2)}} / R=${{r.recall.toFixed(2)}}`;
    get('[data-role="interpretation"]').textContent = 'Precision describes how many detected candidates are useful; recall describes how many true peaks were found.';
    return [
      `The selected detector gives precision ${{r.precision.toFixed(2)}} and recall ${{r.recall.toFixed(2)}}.`,
      'A sensitive detector usually improves recall but increases false positives and review workload.',
      'For quantitative workflows, AI-supported peak detection should be validated as a review aid or decision-support tool, not treated as an unchecked integration authority.'
    ];
  }}

  function drawRetention() {{
    const r = payload.retention[retentionKey()]; clear();
    const all = r.true.concat(r.pred); const mp = mapper(all, all, 58, 26, 28, 50); grid(mp, 'Observed RT (min)', 'Predicted RT (min)');
    line(mp, [mp.xmin, mp.xmax], [mp.xmin, mp.xmax], colors.magenta, 2);
    scatter(mp, r.true, r.pred, colors.cyan, 4.3);
    setLegend([{{color:colors.magenta,label:'ideal'}}, {{color:colors.cyan,label:'test compounds'}}]);
    get('[data-role="description"]').textContent = 'Retention-time prediction';
    get('[data-role="metric"]').textContent = `${{r.mae.toFixed(2)}} min`;
    get('[data-role="interpretation"]').textContent = 'Mean absolute error after the selected training and calibration strategy.';
    return [
      `The selected model gives a retention-time MAE of ${{r.mae.toFixed(2)}} min.`,
      'Increasing training-set size improves the learned structure-property trend, while calibrants correct method- or instrument-specific shifts.',
      'Retention-time predictions are most useful as orthogonal evidence; they should be combined with spectral and chemical plausibility information.'
    ];
  }}

  function drawIdentification() {{
    const r = payload.identification[identificationKey()]; clear();
    const labels = r.names.slice(0,10), scores = r.score.slice(0,10);
    const left=150, top=28, w=canvas.width-left-24, barH=26, gap=9;
    ctx.font='12px system-ui'; ctx.fillStyle=colors.text; ctx.fillText('Top candidates by combined score', left, 16);
    const maxScore = Math.max(...scores, 1);
    labels.forEach((name,i)=>{{
      const y = top+i*(barH+gap); const bw = scores[i]/maxScore*w;
      ctx.fillStyle = name === r.true_name ? colors.green : colors.cyan;
      ctx.fillRect(left, y, bw, barH);
      ctx.strokeStyle = '#fff'; ctx.strokeRect(left, y, bw, barH);
      ctx.fillStyle = colors.text; ctx.fillText(name, 10, y+17); ctx.fillText(scores[i].toFixed(2), left+bw+8, y+17);
    }});
    setLegend([{{color:colors.cyan,label:'candidate'}}, {{color:colors.green,label:'simulated true identity'}}]);
    get('[data-role="description"]').textContent = 'Identification scoring';
    get('[data-role="metric"]').textContent = labels[0];
    get('[data-role="interpretation"]').textContent = `Top-ranked candidate. The simulated true candidate is ${{r.true_name}}; observed RT is ${{r.true_rt.toFixed(2)}} min.`;
    return [
      `The top-ranked candidate is ${{labels[0]}}; the simulated true candidate is ${{r.true_name}}.`,
      'A balanced score can demote spectral decoys whose retention time is chemically implausible.',
      'This is decision support: identification confidence still depends on reference quality, RT uncertainty, adduct/isotope logic, and validation rules.'
    ];
  }}

  let currentAnswers = [];
  let currentQuestions = [];
  function update() {{
    const module = control('module');
    getAll('[data-panel]').forEach(p => p.classList.toggle('hidden', p.getAttribute('data-panel') !== module));
    currentQuestions = questionSets[module] || [];
    if (module === 'baseline') currentAnswers = drawBaseline();
    if (module === 'peaks') currentAnswers = drawPeaks();
    if (module === 'retention') currentAnswers = drawRetention();
    if (module === 'identification') currentAnswers = drawIdentification();
    get('[data-role="questions"]').innerHTML = '<strong>Questions for this application module</strong><ol>' + currentQuestions.map(q => `<li>${{q}}</li>`).join('') + '</ol>';
    get('[data-role="suggested"]').innerHTML = '<strong>Suggested answers</strong><ol>' + currentAnswers.map((a, i) => `<li><strong>Q${{i+1}}.</strong> ${{a}}</li>`).join('') + '</ol>';
  }}

  getAll('select').forEach(el => el.addEventListener('change', update));
  get('[data-action="showAnswers"]').addEventListener('click', () => {{ const box=get('[data-role="suggested"]'); box.style.display = box.style.display === 'block' ? 'none' : 'block'; }});
  get('[data-action="useAnswers"]').addEventListener('click', () => {{ getAll('textarea').forEach((ta,i)=>{{ ta.value = currentAnswers[i] || ''; }}); }});
  get('[data-action="copy"]').addEventListener('click', async () => {{
    const text = getAll('textarea').map((ta,i)=>`Answer ${{i+1}}: ${{ta.value}}`).join('\\n\\n');
    try {{ await navigator.clipboard.writeText(text); }} catch (e) {{ console.log(e); }}
  }});
  get('[data-action="download"]').addEventListener('click', () => {{
    const text = 'HPLC 2026 - Applications of AI in Chromatography\\nDr. Tijmen S. Bos / IDEAS / bos-ideas.com\\n\\n' + getAll('textarea').map((ta,i)=>`Answer ${{i+1}}: ${{ta.value}}`).join('\\n\\n');
    const blob = new Blob([text], {{type:'text/plain'}}); const url = URL.createObjectURL(blob);
    const a = document.createElement('a'); a.href = url; a.download = 'hplc2026_applications_answers.txt'; a.click(); URL.revokeObjectURL(url);
  }});
  update();
}})();
</script>
"""

display(HTML(html))


## 9. Student-provided structures and chromatograms

This section lets students provide their own simple chemical structures and/or chromatograms directly in the browser.

For structures, paste one or more lines as either `name,SMILES` or just `SMILES`. The descriptor calculation is intentionally simple and transparent. It is **not** RDKit, not a validated QSRR model, and not a replacement for a trained model on a defined method/column/solvent system.

For chromatograms, paste CSV text with two columns: `time,signal`. The analyzer performs simple baseline estimation, smoothing, and peak-candidate detection. It is suitable for teaching and rapid inspection, not for validated quantitative integration.


The interactive panel now shows three explicit questions for the selected input type. The suggested answers map one-to-one to these questions; students should adapt them to their own pasted structures or chromatogram.


In [ ]:
# Student-provided structure and chromatogram analyzer.
# Browser-side, no ipywidgets, no external packages, no RDKit.

from IPython.display import HTML, display
import uuid, json

student_container_id = "student_input_app_" + uuid.uuid4().hex[:8]
default_csv = '''time,signal
0.00,37
0.10,38
0.20,39
0.30,40
0.40,42
0.50,43
0.60,44
0.70,43
0.80,45
0.90,49
1.00,58
1.10,88
1.20,132
1.30,90
1.40,60
1.50,51
1.60,49
1.70,50
1.80,51
1.90,52
2.00,54
2.10,55
2.20,57
2.30,61
2.40,77
2.50,124
2.60,201
2.70,243
2.80,174
2.90,92
3.00,64
3.10,59
3.20,58
3.30,58
3.40,59
3.50,60
3.60,61
3.70,62
3.80,63
3.90,64
4.00,66
4.10,69
4.20,77
4.30,109
4.40,171
4.50,224
4.60,190
4.70,122
4.80,83
4.90,73
5.00,71
5.10,72
5.20,74
5.30,76
5.40,77
5.50,78
5.60,79
5.70,80
5.80,81
5.90,83
6.00,84
6.10,85
6.20,86
6.30,87
6.40,88
6.50,89
6.60,90
6.70,91
6.80,92
6.90,93
7.00,94
'''
default_structures = '''caffeine,Cn1cnc2n(C)c(=O)n(C)c(=O)c12
aspirin,CC(=O)OC1=CC=CC=C1C(=O)O
ibuprofen,CC(C)CC1=CC=C(C=C1)C(C)C(=O)O
glycine,NCC(=O)O
benzene,c1ccccc1'''

html = r'''
<div id="__CID__" class="student-input-dashboard">
  <style>
    #__CID__ {
      --cyan:#43CEFF; --magenta:#DB3FC8; --gold:#FFC000; --green:#92D050;
      --text:#16495A; --grid:#E5F8FD; --pale-gold:#FFF2BF;
      font-family:system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;
      color:var(--text); width:100%; max-width:100%; min-width:0;
      border:2px solid var(--magenta); border-radius:18px; padding:14px;
      box-sizing:border-box; background:#ffffff; overflow:visible!important;
    }
    #__CID__ *{box-sizing:border-box;min-width:0}
    #__CID__ .title{font-size:20px;font-weight:800;margin-bottom:4px}
    #__CID__ .subtitle{font-size:13px;margin-bottom:14px;opacity:.85;line-height:1.35}
    #__CID__ .layout{
      display:grid;
      grid-template-columns:minmax(230px,300px) minmax(0,1fr);
      gap:12px;
      align-items:start;
      width:100%;
    }
    #__CID__ .card{border:1px solid #d8eef5;border-radius:14px;padding:12px;background:#fbfdfe;box-sizing:border-box;min-width:0;overflow-wrap:anywhere}
    #__CID__ .plot-card{min-width:0;overflow:hidden}
    #__CID__ .answer-card{grid-column:1 / -1;display:grid;grid-template-columns:minmax(220px,.9fr) minmax(260px,1.1fr);gap:10px;align-items:start}
    #__CID__ .answer-top{min-width:0}
    #__CID__ .answer-boxes{min-width:0}
    #__CID__ label{display:block;font-weight:700;font-size:12px;margin-top:10px;margin-bottom:4px}
    #__CID__ select{width:100%;max-width:100%}
    #__CID__ textarea{width:100%;max-width:100%;min-height:128px;box-sizing:border-box;border-radius:10px;border:1px solid #c9dfe7;padding:8px;font-family:ui-monospace,Menlo,Consolas,monospace;font-size:11.5px;resize:vertical}
    #__CID__ canvas{width:100%;max-width:100%;height:auto;display:block;border:1px solid #d8eef5;border-radius:14px;background:white}
    #__CID__ button{border:none;border-radius:10px;padding:8px 10px;margin:4px 4px 4px 0;font-weight:700;cursor:pointer;color:var(--text);background:var(--cyan)}
    #__CID__ button.secondary{background:var(--gold)}
    #__CID__ .metric{font-size:24px;font-weight:800;color:var(--magenta);margin-top:4px}
    #__CID__ .small{font-size:12px;line-height:1.35}
    #__CID__ .questions{border:2px solid var(--cyan);background:#F4FCFF;border-radius:12px;padding:10px;margin-top:10px;font-size:12px;line-height:1.35}
    #__CID__ .questions ol, #__CID__ .answers ol{margin:6px 0 0 18px;padding:0}
    #__CID__ .question-label{font-weight:700;font-size:12px;margin-top:10px;margin-bottom:4px;line-height:1.3}
    #__CID__ .answers{display:none;border:2px solid var(--gold);background:var(--pale-gold);border-radius:12px;padding:10px;margin-top:10px;font-size:12px;line-height:1.35}
    #__CID__ table{border-collapse:collapse;width:100%;font-size:11px;margin-top:8px;table-layout:auto}
    #__CID__ th,#__CID__ td{border-bottom:1px solid #d8eef5;padding:4px;text-align:left;word-break:break-word}
    #__CID__ .hidden{display:none!important}
    @media(max-width:950px){#__CID__ .layout{grid-template-columns:1fr}#__CID__ .answer-card{grid-column:auto;display:block}#__CID__ canvas{max-height:none}}
  </style>
  <div class="title">Student input application exercise</div><div class="subtitle">Paste a chemical structure list or chromatogram CSV. Processing is local in the browser. The results are teaching estimates and should not be used as validated analytical outputs.</div>
  <div class="layout">
    <div class="card control-card">
      <label>Input type</label><select data-control="mode"><option value="structure">Chemical structures / SMILES</option><option value="chromatogram">Chromatogram CSV</option></select>
      <div data-panel="structure"><label>Structures: one per line as name,SMILES</label><textarea data-input="smiles">__DEFAULT_STRUCTURES__</textarea><label>Stationary-phase context</label><select data-control="phase"><option value="rplc" selected>RPLC-like</option><option value="hilic">HILIC-like</option><option value="mixed">mixed-mode teaching score</option></select></div>
      <div data-panel="chromatogram" class="hidden"><label>Chromatogram CSV: time,signal</label><textarea data-input="csv">__DEFAULT_CSV__</textarea><label>Baseline window</label><select data-control="baselineWindow"><option>7</option><option selected>15</option><option>31</option><option>51</option></select><label>Smoothing window</label><select data-control="smoothWindow"><option>3</option><option selected>7</option><option>15</option><option>25</option></select><label>Detection threshold</label><select data-control="threshold"><option value="2.0">sensitive</option><option value="3.0" selected>balanced</option><option value="5.0">conservative</option></select></div>
      <button data-action="analyze">Analyze input</button><button class="secondary" data-action="restore">Restore example input</button>
    </div>
    <div class="card plot-card"><canvas width="760" height="470" data-role="plot"></canvas><div data-role="table"></div></div>
    <div class="card answer-card">
      <div class="answer-top"><div class="small" data-role="description"></div><div class="metric" data-role="metric"></div><div class="small" data-role="interpretation"></div><div class="questions" data-role="questions"></div><div class="answers" data-role="suggested"></div><button class="secondary" data-action="showAnswers">Show suggested answers</button><button class="secondary" data-action="useAnswers">Use suggested answers in boxes</button></div>
      <div class="answer-boxes"><div class="question-label" data-question-label="0">Question 1</div><textarea data-answer="0" style="min-height:66px;font-family:inherit"></textarea><div class="question-label" data-question-label="1">Question 2</div><textarea data-answer="1" style="min-height:66px;font-family:inherit"></textarea><div class="question-label" data-question-label="2">Question 3</div><textarea data-answer="2" style="min-height:66px;font-family:inherit"></textarea><button data-action="copy">Copy answers</button></div>
    </div>
  </div>
</div>
<script>
(function(){
const root=document.getElementById('__CID__'),canvas=root.querySelector('[data-role="plot"]'),ctx=canvas.getContext('2d');const colors={cyan:'#43CEFF',magenta:'#DB3FC8',gold:'#FFC000',green:'#92D050',text:'#16495A',grid:'#E5F8FD'};const defaultCsv=__DEFAULT_CSV_JSON__,defaultStructures=__DEFAULT_STRUCTURES_JSON__;let currentQuestions=[], currentAnswers=[];function get(s){return root.querySelector(s)}function getAll(s){return Array.from(root.querySelectorAll(s))}['.jp-OutputArea-output','.jp-RenderedHTMLCommon','.output_html','.output_subarea'].forEach(sel=>{const w=root.closest(sel);if(w){w.style.maxHeight='none';w.style.overflow='visible'}});function clear(){ctx.clearRect(0,0,canvas.width,canvas.height);ctx.fillStyle='white';ctx.fillRect(0,0,canvas.width,canvas.height)}function mapper(xs,ys,padL=58,padT=24,padR=25,padB=52){const xmin=Math.min(...xs),xmax=Math.max(...xs),ymin=Math.min(...ys),ymax=Math.max(...ys),xr=(xmax-xmin)||1,yr=(ymax-ymin)||1;return{padL,padT,padR,padB,x:x=>padL+(x-(xmin-.04*xr))/((xmax+.04*xr)-(xmin-.04*xr))*(canvas.width-padL-padR),y:y=>canvas.height-padB-(y-(ymin-.08*yr))/((ymax+.12*yr)-(ymin-.08*yr))*(canvas.height-padT-padB)}}function grid(mp,xlab,ylab){ctx.strokeStyle=colors.grid;ctx.lineWidth=1;ctx.font='12px system-ui';ctx.fillStyle=colors.text;for(let i=0;i<=5;i++){let x=mp.padL+i*(canvas.width-mp.padL-mp.padR)/5;ctx.beginPath();ctx.moveTo(x,mp.padT);ctx.lineTo(x,canvas.height-mp.padB);ctx.stroke()}for(let i=0;i<=5;i++){let y=mp.padT+i*(canvas.height-mp.padT-mp.padB)/5;ctx.beginPath();ctx.moveTo(mp.padL,y);ctx.lineTo(canvas.width-mp.padR,y);ctx.stroke()}ctx.strokeStyle=colors.text;ctx.strokeRect(mp.padL,mp.padT,canvas.width-mp.padL-mp.padR,canvas.height-mp.padT-mp.padB);ctx.fillText(xlab,canvas.width/2-45,canvas.height-15);ctx.save();ctx.translate(16,canvas.height/2+55);ctx.rotate(-Math.PI/2);ctx.fillText(ylab,0,0);ctx.restore()}function line(mp,xs,ys,color,lw=2){ctx.strokeStyle=color;ctx.lineWidth=lw;ctx.beginPath();xs.forEach((x,i)=>{const X=mp.x(x),Y=mp.y(ys[i]);if(i===0)ctx.moveTo(X,Y);else ctx.lineTo(X,Y)});ctx.stroke()}function parseCsv(text){const rows=text.trim().split(/\n+/).map(r=>r.split(/[;,\t ]+/).filter(Boolean)),out=[];rows.forEach((r,idx)=>{if(idx===0&&isNaN(Number(r[0])))return;if(r.length>=2){const t=Number(r[0]),y=Number(r[1]);if(Number.isFinite(t)&&Number.isFinite(y))out.push([t,y])}});return out}function movingAverage(a,w){w=Math.max(1,Number(w)|0);const h=Math.floor(w/2);return a.map((_,i)=>{let s=0,n=0;for(let j=i-h;j<=i+h;j++){if(j>=0&&j<a.length){s+=a[j];n++}}return s/n})}function movingMin(a,w){w=Math.max(1,Number(w)|0);const h=Math.floor(w/2);return a.map((_,i)=>{let m=Infinity;for(let j=i-h;j<=i+h;j++){if(j>=0&&j<a.length)m=Math.min(m,a[j])}return m})}function median(a){const b=a.slice().sort((x,y)=>x-y);return b[Math.floor(b.length/2)]||0}function mad(a){const m=median(a);return median(a.map(x=>Math.abs(x-m)))||1}function analyzeChromatogram(){const rows=parseCsv(get('[data-input="csv"]').value);clear();if(rows.length<8){get('[data-role="description"]').textContent='Chromatogram input error';get('[data-role="metric"]').textContent='Need CSV';return}const t=rows.map(r=>r[0]),y=rows.map(r=>r[1]),bw=Number(get('[data-control="baselineWindow"]').value),sw=Number(get('[data-control="smoothWindow"]').value),thr=Number(get('[data-control="threshold"]').value),base=movingAverage(movingMin(y,bw),bw),corr=y.map((v,i)=>v-base[i]),sigma=1.4826*mad(corr),threshold=thr*sigma,peaks=[];for(let i=1;i<corr.length-1;i++){if(corr[i]>threshold&&corr[i]>=corr[i-1]&&corr[i]>=corr[i+1]){if(peaks.length&&t[i]-peaks[peaks.length-1].rt<.18){if(corr[i]>peaks[peaks.length-1].height)peaks[peaks.length-1]={rt:t[i],height:corr[i]}}else peaks.push({rt:t[i],height:corr[i]})}}const mp=mapper(t,y.concat(base),58,25,26,52);grid(mp,'Time (min)','Signal');line(mp,t,y,colors.text,1.4);line(mp,t,base,colors.green,2.1);peaks.forEach(p=>{const idx=t.reduce((best,_,i)=>Math.abs(t[i]-p.rt)<Math.abs(t[best]-p.rt)?i:best,0);ctx.fillStyle=colors.magenta;ctx.beginPath();ctx.arc(mp.x(p.rt),mp.y(y[idx]),5,0,2*Math.PI);ctx.fill();ctx.strokeStyle='white';ctx.stroke()});get('[data-role="description"]').textContent='User chromatogram analysis';get('[data-role="metric"]').textContent=peaks.length+' peaks';get('[data-role="interpretation"]').textContent=`Estimated noise ${sigma.toFixed(2)}; threshold ${threshold.toFixed(2)}. Peaks are candidate local maxima after baseline correction.`;get('[data-role="table"]').innerHTML='<table><tr><th>#</th><th>RT</th><th>height above baseline</th></tr>'+peaks.map((p,i)=>`<tr><td>${i+1}</td><td>${p.rt.toFixed(2)}</td><td>${p.height.toFixed(1)}</td></tr>`).join('')+'</table>';currentQuestions=[
  'What does the algorithm find in the pasted chromatogram?',
  'How do the baseline, smoothing, and threshold settings influence the result?',
  'What validation would be required before using this result quantitatively?'
];currentAnswers=[`The pasted chromatogram produced ${peaks.length} candidate peaks using the selected baseline and threshold settings.`,'Changing baseline and smoothing windows can change broad-peak treatment, noise suppression, and whether weak shoulders are counted as peaks.','For quantitative use, the method would need validation against known samples, integration rules, uncertainty estimates, and manual-review criteria.'];updateAnswers()}function atomCount(smiles,token){return(smiles.match(new RegExp(token,'g'))||[]).length}function describeSmiles(smiles){const s=smiles.trim(),cl=atomCount(s,'Cl'),br=atomCount(s,'Br'),clean=s.replace(/Cl/g,'').replace(/Br/g,''),c=(clean.match(/C|c/g)||[]).length,n=(clean.match(/N|n/g)||[]).length,o=(clean.match(/O|o/g)||[]).length,su=(clean.match(/S|s/g)||[]).length,p=(clean.match(/P|p/g)||[]).length,hal=cl+br+(clean.match(/F|I/g)||[]).length,rings=new Set((clean.match(/[1-9]/g)||[])).size,branches=(clean.match(/\(/g)||[]).length,hetero=n+o+su+p,mass=12*c+14*n+16*o+32.1*su+31*p+35.5*cl+79.9*br+19*((clean.match(/F/g)||[]).length)+127*((clean.match(/I/g)||[]).length),logp=.42*c+.35*hal+.18*rings-.72*hetero-.12*branches,psa=13*n+17*o+25*su+30*p,hbond=n+o+su,rplc=1.2+.62*logp+.006*mass-.018*psa+.12*rings,hilic=1.1+.055*psa+.42*hbond-.33*logp;return{mass,logp,psa,hbond,rplc:Math.max(.2,Math.min(12,rplc)),hilic:Math.max(.2,Math.min(12,hilic))}}function parseSmilesLines(text){return text.trim().split(/\n+/).map((line,i)=>{const parts=line.split(/,(.+)/);if(parts.length>=3)return{name:parts[0].trim()||`compound_${i+1}`,smiles:parts[1].trim()};return{name:`compound_${i+1}`,smiles:line.trim()}}).filter(x=>x.smiles)}function analyzeStructures(){const items=parseSmilesLines(get('[data-input="smiles"]').value);clear();if(!items.length){get('[data-role="description"]').textContent='SMILES input error';get('[data-role="metric"]').textContent='No structures';return}const rows=items.map(x=>Object.assign({},x,describeSmiles(x.smiles))),phase=get('[data-control="phase"]').value,scores=rows.map(r=>phase==='hilic'?r.hilic:(phase==='mixed'?(r.rplc+r.hilic)/2:r.rplc)),left=150,top=28,barH=26,gap=10,maxScore=Math.max(...scores,1),w=canvas.width-left-35;ctx.font='12px system-ui';ctx.fillStyle=colors.text;ctx.fillText('Teaching retention score from simple SMILES descriptors',left,18);rows.forEach((r,i)=>{const y=top+i*(barH+gap),color=i%4===0?colors.cyan:i%4===1?colors.magenta:i%4===2?colors.gold:colors.green;ctx.fillStyle=color;ctx.fillRect(left,y,w*scores[i]/maxScore,barH);ctx.fillStyle=colors.text;ctx.fillText(r.name.slice(0,22),12,y+17);ctx.fillText(scores[i].toFixed(2),left+w*scores[i]/maxScore+8,y+17)});const best=rows[scores.indexOf(Math.max(...scores))];get('[data-role="description"]').textContent='User structure / SMILES analysis';get('[data-role="metric"]').textContent=best.name;get('[data-role="interpretation"]').textContent=`Highest ${phase.toUpperCase()} teaching-retention score. Descriptors are approximate string-derived proxies, not validated molecular descriptors.`;get('[data-role="table"]').innerHTML='<table><tr><th>compound</th><th>logP proxy</th><th>PSA proxy</th><th>mass proxy</th><th>RPLC RT</th><th>HILIC RT</th></tr>'+rows.map(r=>`<tr><td>${r.name}</td><td>${r.logp.toFixed(2)}</td><td>${r.psa.toFixed(0)}</td><td>${r.mass.toFixed(1)}</td><td>${r.rplc.toFixed(2)}</td><td>${r.hilic.toFixed(2)}</td></tr>`).join('')+'</table>';currentQuestions=[
  'Which compound is predicted to have the strongest retention under the selected phase context?',
  'Which molecular features appear to drive the prediction?',
  'What would be needed to turn this teaching model into a defensible QSRR model?'
];currentAnswers=[`The SMILES-derived descriptors give ${best.name} the highest score under the selected phase context.`,'The calculation is chemically interpretable but simplified: it mainly responds to carbon count, hetero atoms, polarity, rings, and approximate mass.','A real QSRR model would need validated descriptors, a defined LC method, representative calibration data, calibration transfer checks, and external validation.'];updateAnswers()}function updateAnswers(){
  get('[data-role="questions"]').innerHTML='<strong>Questions for this input</strong><ol>'+currentQuestions.map(q=>`<li>${q}</li>`).join('')+'</ol>';
  get('[data-role="suggested"]').innerHTML='<strong>Suggested answers</strong><ol>'+currentAnswers.map((a,i)=>`<li><strong>Q${i+1}.</strong> ${a}</li>`).join('')+'</ol>';
  getAll('[data-question-label]').forEach((el,i)=>{el.textContent=`Question ${i+1}: ${currentQuestions[i]||''}`});
}function updateMode(){const m=get('[data-control="mode"]').value;getAll('[data-panel]').forEach(p=>p.classList.toggle('hidden',p.getAttribute('data-panel')!==m));get('[data-role="table"]').innerHTML='';if(m==='structure')analyzeStructures();else analyzeChromatogram()}get('[data-control="mode"]').addEventListener('change',updateMode);getAll('select').forEach(el=>el.addEventListener('change',()=>{if(get('[data-control="mode"]').value==='structure')analyzeStructures();else analyzeChromatogram()}));get('[data-action="analyze"]').addEventListener('click',()=>{if(get('[data-control="mode"]').value==='structure')analyzeStructures();else analyzeChromatogram()});get('[data-action="restore"]').addEventListener('click',()=>{get('[data-input="csv"]').value=defaultCsv;get('[data-input="smiles"]').value=defaultStructures;updateMode()});get('[data-action="showAnswers"]').addEventListener('click',()=>{const box=get('[data-role="suggested"]');box.style.display=box.style.display==='block'?'none':'block'});get('[data-action="useAnswers"]').addEventListener('click',()=>{getAll('[data-answer]').forEach((ta,i)=>ta.value=currentAnswers[i]||'')});get('[data-action="copy"]').addEventListener('click',async()=>{const text=getAll('[data-answer]').map((ta,i)=>`Question ${i+1}: ${currentQuestions[i]||''}\nAnswer ${i+1}: ${ta.value}`).join('\n\n');try{await navigator.clipboard.writeText(text)}catch(e){console.log(e)}});updateMode();})();
</script>
'''
html = html.replace('__CID__', student_container_id)
html = html.replace('__DEFAULT_CSV__', default_csv.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;'))
html = html.replace('__DEFAULT_STRUCTURES__', default_structures.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;'))
html = html.replace('__DEFAULT_CSV_JSON__', json.dumps(default_csv))
html = html.replace('__DEFAULT_STRUCTURES_JSON__', json.dumps(default_structures))
display(HTML(html))


## 10. Take-home interpretation

AI in chromatography is not one thing. The useful question is not “Should I use AI?” but **which part of the chromatographic workflow has a decision, prediction, or pattern-recognition problem that can be stated clearly and validated properly?**

For data processing, the critical issue is whether the model learns the chemical signal or the background artefact. For retention-time prediction and identification, calibration and uncertainty matter as much as the model architecture. For optimization, data efficiency and a well-designed response function are often more important than raw model complexity.

The practical message is therefore consistent with the course: AI can be useful, but the chromatographic problem definition, data quality, validation strategy, and human expertise determine whether it is actually useful.
